### Section B

###B1

In [0]:
dbutils.widgets.text("catalog", "vp_ms20010501")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"create table if not exists {catalog}.bronze.stations")

spark.sql(f"""
          copy into {catalog}.bronze.stations
          from (
            select *, _metadata.file_name as _source_file, current_timestamp() as _ingested_at
            from '/Volumes/{catalog}/landing/raw/stations/'
          )
          fileformat = CSV
          format_options ("header" = "true", "inferschema" = "false")
          copy_options ("mergeSchema" = "true")
          """)

In [0]:
spark.sql(f"create table if not exists {catalog}.bronze.tariffs")

spark.sql(f"""
          copy into {catalog}.bronze.tariffs
          from (
            select *, _metadata.file_name as _source_file, current_timestamp() as _ingested_at
            from '/Volumes/{catalog}/landing/raw/tariffs/'
          )  
          fileformat = CSV
          format_options ("header" = "true", "inferschema" = "false")
          copy_options ("mergeSchema" = "true")
          """)

###B2

In [0]:
spark.sql(f"create table if not exists {catalog}.bronze.members")

spark.sql(f"""
          copy into {catalog}.bronze.members
          from (
            select *, cast(null as date) as change_effective_date, _metadata.file_name as _source_file, current_timestamp() as _ingested_at
            from '/Volumes/{catalog}/landing/raw/members/'
          )  
          fileformat = CSV
          format_options ("header" = "true", "inferschema" = "false")
          copy_options ("mergeSchema" = "true")
          """)

###B3

In [0]:
spark.sql(f"create table if not exists {catalog}.bronze.chargers")

spark.sql(f"""
          copy into {catalog}.bronze.chargers
          from (
            select *, _metadata.file_name as _source_file, current_timestamp() as _ingested_at
            from '/Volumes/{catalog}/landing/raw/chargers/'
          )  
          fileformat = JSON
          format_options ("multiline" = "true", "mergeSchema" = "true")
          copy_options ("mergeSchema" = "true")
          """)

###B4

In [0]:
dbutils.widgets.text("checkpoint_name", "sessions")
dbutils.widgets.text("sessions_schema", "sessions_schema")

checkpoint_name = dbutils.widgets.get("checkpoint_name")
schema_name = dbutils.widgets.get("sessions_schema")

In [0]:
spark.sql(f"create volume if not exists {catalog}.ops.{checkpoint_name}")

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")
    .option("cloudFiles.inferSchema", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.schemaLocation", f"/Volumes/{catalog}/ops/{checkpoint_name}/{schema_name}") 
    .load(f"/Volumes/{catalog}/landing/raw/sessions/")
    .selectExpr("*", "_metadata.file_name as _source_file", "current_timestamp() as _ingested_at")
    .writeStream
    .trigger(availableNow=True)
    .option("checkpointLocation", f"/Volumes/{catalog}/ops/pipeline_state/{checkpoint_name}")
    .option("mergeSchema", "true")
    .table(f"{catalog}.bronze.sessions"))
